In [ ]:
import yaml

from sim.drive_simulator import (
    CarSim,
    Commander,
)
from sim.vehicle import VehicleProp
from sim.mission_base import MissionBase
from sim.goal import GoalCircle
from sim.drawer import VehicleDrawer
from sim.sign import Sign
from world.type_b_world import type_b_circuit

with open("config/type-b.yaml", "r") as f:
    vehicle_config = yaml.safe_load(f)

prop = VehicleProp(**vehicle_config)
sim = CarSim(prop)
com = Commander(sim)

move = com.move
rotate = com.rotate
camera = com.camera
search = com.search
search_all = com.search_all
wait = com.wait

In [ ]:
class Mission3(MissionBase):
    def __init__(self):
        super().__init__(type_b_circuit, t_max=80)
        self.goals = [
            GoalCircle((2, 0.0), 0.2, should_stop=False),
            GoalCircle((4.3, 1.0), 0.2, should_stop=False),
            GoalCircle((3.2, 2.2), 0.2, should_stop=False),
            GoalCircle((1.7, 1.0), 0.2, should_stop=False),
            GoalCircle((0.0, 0.8), 0.2, should_stop=False),
            GoalCircle((0.0, 0.0), 0.2),
        ]
        self.set_signs(
            [
                Sign(x=1.1, y=-0.1, name="sign2"),
                Sign(x=2.2, y=-0.1, name="sign2"),
                Sign(x=3.3, y=-0.1, name="sign2"),
                Sign(x=4.1, y=0.3, name="sign2"),
                Sign(x=4.5, y=1.3, name="sign2"),
                Sign(x=3.9, y=2.3, name="sign2"),
                Sign(x=2.8, y=2.3, name="sign2"),
                Sign(x=3.1, y=1.3, name="sign3"),
                Sign(x=2.2, y=1.0, name="sign2"),
                Sign(x=1.2, y=0.9, name="sign2"),
                Sign(x=0.2, y=0.8, name="sign2"),
                Sign(x=-0.6, y=0.4, name="sign2"),
                Sign(x=0.2, y=-0.2, name="sign1"),
            ]
        )

    def command_func(self):
        """回答例１：
        回転と直進を使用して標識に近づく
        - sign2を最後に見た後に標識を見失ったら反時計回りに回転する
        - sign3を最後に見た後に標識を見失ったら時計回りに回転する
        """
        last_name = None
        while True:
            pos = search()
            if pos is None:
                if last_name == "sign2":
                    rotate(w=45)
                elif last_name == "sign3":
                    rotate(w=-45)
                else:
                    move(v=0)
            else:
                last_name = pos.name
                if pos.theta > 5:
                    rotate(w=45)
                elif pos.theta < -5:
                    rotate(w=-45)
                else:
                    move(v=1)


sim.set_mission(Mission3())
sim.run()
drawer = VehicleDrawer(sim)
drawer.show()